<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-10-tuning-and-evaluation/lesson-10.1-sft-lora/notebooks/GCP_Capstone_10.1_SFT_LoRA.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.1 SFT with LoRA — The Dataset Is a Document, the Model Is a Setting, the Gate Is the Judge
**Netsetos GenAI Engineering — GCP Capstone** · Module 10 · rebuilt on the live lane, 9 September 2026

Managed supervised fine-tuning on Vertex AI, done the way a lane can defend it. The dataset comes from the corpus the lane already chunks - one question and answer per real chunk, in the lane's own citation grammar - built by the kit's `make trainset` (or here, smaller), scanned by the one PII list, with every golden question excluded. The job is launched behind a switch, on a base the service actually accepts. And the judgement is the gate: the same 65 rows and nine thresholds, on the live revision and on a candidate revision with the tuned endpoint behind it. Nothing above the API changes.


## Setup
The kit, the roster member, and three switches: `LAUNCH` (a billed job), `TUNED_ENDPOINT` (paste what `make tune` printed), `ROWS` (the notebook builds a small set).


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-dlp==3.39.0 google-cloud-firestore==2.30.0 fastapi==0.141.1 pydantic-settings==2.15.0 pandas==2.3.3 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "rag-production-hardening"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
LAUNCH         = False   # flip to submit the managed tuning job (billed per training token): an explicit act
TUNED_ENDPOINT = ""      # paste what `make tune` printed (projects/.../endpoints/...) to run the A/B cells
ROWS           = 40      # the notebook builds a small set; `make trainset` builds the frozen 300-row one

print("kit:", KIT, "| API:", API_URL, "| datasets:", f"gs://{DATASETS}/sft/")


## Cell 1: The API, the usage rows, the kit's builders


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, CALLED THE WAY THE UI CALLS IT: one ID token per request, minted AS the roster member,
# audience = the API (7.3's hour-long fuse never arms). The kit mints it (documind_tools._id_token).
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). They
# land in Cloud Logging first (the sink copies them to BigQuery); this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

def gcs_text(uri: str) -> str:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_text()

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: make_trainset, judge, tune, run_eval
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules: cache_manager, router, breakers, cost
print("helpers: api(), usage_rows(), gcs_text(); the kit's evals/ and rag-api/ on sys.path")


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


## Cell 2: The dataset is a document
The frozen file from the datasets bucket when `make trainset` has run and its rows were reviewed; otherwise the same builder here, on a smaller sample. Either way: the corpus, the lane's grammar, the golden set excluded, DLP clean, a manifest.


In [ ]:
import make_trainset as mt

# THE DATASET IS A DOCUMENT. Not the logs (sink.tf keeps question and answer out of BigQuery), not the
# answer cache (on the lane it holds the golden questions every eval run asked - the TEST set). The corpus:
# the real Acts and the synthetic handbooks the lane already chunks, one question and one answer per
# chunk in the lane's own grammar - the generator's SYSTEM, one [Source 1], ModelDraft's JSON as the
# target - so the habit taught is the habit served. The kit's builder does it; this cell calls it.
frozen = f"gs://{DATASETS}/sft/documind_sft_v1.manifest.json"
try:
    manifest = json.loads(gcs_text(frozen))
    print("the frozen dataset is in the bucket - make trainset ran, the rows were reviewed and committed:")
    print(json.dumps({k: manifest[k] for k in ("version", "built_at", "rows", "refusals", "dropped_golden_overlap", "dropped_pii")}, indent=1))
    VERTEX_JSONL = gcs_text(f"gs://{DATASETS}/sft/documind_sft_v1.vertex.jsonl")
    open("documind_sft.vertex.jsonl", "w", encoding="utf-8").write(VERTEX_JSONL)
    SOURCE = "frozen"
except Exception as e:
    print(f"no frozen dataset in gs://{DATASETS}/sft/ ({type(e).__name__}) - building {ROWS} rows here, the same way:")
    chunks = mt.sample(mt.load_chunks(TENANT, PROJECT_ID), ROWS)
    rows = mt.ask_pairs(PROJECT_ID, chunks)                               # gemini-3.6-flash, one structured call per chunk
    golden = [json.loads(l) for l in open(f"{KIT}/deploy/evals/golden.jsonl", encoding="utf-8") if l.strip()]
    rows, dropped_golden = mt.exclude_golden(rows, golden)               # the test set never enters the file
    rows, dropped_pii = mt.redact(rows)                                   # shared/pii: weights cannot be filtered later
    manifest = mt.write(rows, "sft", "colab", TENANT, dropped_golden, dropped_pii)
    VERTEX_JSONL = open("sft/documind_sft_colab.vertex.jsonl", encoding="utf-8").read()
    open("documind_sft.vertex.jsonl", "w", encoding="utf-8").write(VERTEX_JSONL)
    SOURCE = "colab"
    print(json.dumps({k: manifest[k] for k in ("rows", "refusals", "dropped_golden_overlap", "dropped_golden_ids", "dropped_pii")}, indent=1))

examples = [json.loads(l) for l in VERTEX_JSONL.splitlines() if l.strip()]
assert len(examples) >= 20, "fewer than twenty rows: the corpus mirrors are missing (git clone --depth 1 keeps them) or every row was dropped"
ex0 = examples[0]["contents"]
print(f"\n{len(examples)} rows ({SOURCE}). One example:")
print("  user  :", ex0[0]["parts"][0]["text"][-220:].replace("\n", " | "))
print("  model :", ex0[1]["parts"][0]["text"][:200])


## Cell 3: The two rules, re-run on the file
The format check is the cheap half. Zero PII findings and zero golden overlap are the rules that decide whether the file may be used.


In [ ]:
from shared.documind_schemas import ModelDraft
from shared import pii

# TWO RULES, NOT A FORMAT CHECK. The format check is the cheap half (roles alternate, the last turn is the
# model's, parts carry text). The two that decide whether the file may be used are: zero PII findings
# through the ONE list (shared/pii.py - a PAN in the weights is a PAN every tenant can be handed), and
# zero overlap with the golden set (the test set). The builder applied both; re-run them on the file,
# because a rule applied once and never re-checked is a rule you are trusting.
def validate(examples: list[dict]) -> list[str]:
    errors = []
    for i, rec in enumerate(examples, 1):
        roles = [c.get("role") for c in rec.get("contents", [])]
        if roles != ["user", "model"]:
            errors.append(f"row {i}: roles {roles}, expected user then model")
        if any("text" not in part for c in rec.get("contents", []) for part in c.get("parts", [])):
            errors.append(f"row {i}: a part without text")
        try:
            ModelDraft.model_validate_json(rec["contents"][1]["parts"][0]["text"])
        except Exception as e:
            errors.append(f"row {i}: the target is not ModelDraft's JSON ({type(e).__name__})")
    return errors

problems = validate(examples)
print(f"format: {len(problems)} problems")
assert not problems, problems[:3]

golden = [json.loads(l) for l in open(f"{KIT}/deploy/evals/golden.jsonl", encoding="utf-8") if l.strip()]
as_rows = [{"chunk_id": f"{TENANT}:x", "question": e["contents"][0]["parts"][0]["text"].rsplit("Question: ", 1)[-1],
            "text": e["contents"][0]["parts"][0]["text"].split("Context:", 1)[-1].rsplit("Question:", 1)[0]} for e in examples]
_, overlap = mt.exclude_golden(as_rows, golden)
print(f"golden overlap on the file: {len(overlap)} rows")
assert not overlap, [(o["overlaps"], o["rule"], o["question"][:60]) for o in overlap[:3]]

texts = [e["contents"][0]["parts"][0]["text"].rsplit("Question: ", 1)[-1] for e in examples] + [e["contents"][1]["parts"][0]["text"] for e in examples]
findings = pii.inspect_many(texts)                       # the same request shape the ingest worker sends
hits = [(i, f) for i, f in enumerate(findings) if f]
print(f"DLP on {len(texts)} questions and answers: {len(hits)} findings")
assert not hits, hits[:3]
print("\nthe file may be used: the format holds, the test set is not in it, and DLP found nothing")


## Cell 4: The price, from the file


In [ ]:
# THE PRICE, FROM THE FILE - not from a table. count_tokens on a sample of rows gives tokens per example;
# rows x epochs x that is the training-token bill. The per-token rate is the tuning page's (verified
# 2026-09-04 for 10.1; re-verify - it moves), and inference on a tuned model costs the BASE model's
# rate: the adapter is free to serve, which is what makes LoRA the cheap kind of tuning.
EPOCHS, ADAPTER = 3, 4
SFT_USD_PER_1M = 3.50            # managed SFT, per 1M training tokens (10.1, 4 Sept 2026) - re-verify on the pricing page
sample = examples[:20]
tokens = [gen.models.count_tokens(model="gemini-3.6-flash",
                                  contents=[e["contents"][0]["parts"][0]["text"], e["contents"][1]["parts"][0]["text"]]).total_tokens
          for e in sample]
per_example = sum(tokens) / len(tokens)
training_tokens = per_example * len(examples) * EPOCHS
usd = training_tokens * SFT_USD_PER_1M / 1_000_000
print(f"tokens per example (measured on {len(sample)}): {per_example:,.0f}")
print(f"{len(examples)} rows x {EPOCHS} epochs = {training_tokens:,.0f} training tokens = ${usd:.2f} = Rs {usd * 85:,.0f}")
print("inference afterwards: the base model's rate - cost.py prices a tuned endpoint through RAG_MODEL_BASE")


## Cell 5: The base-model check, then the launch
The kit's `tune.py` refuses a base managed SFT does not accept, before anything is uploaded or paid for. `LAUNCH=False` ships; `make tune` is the explicit act.


In [ ]:
import tune as kit_tune

# THE BASE-MODEL CHECK IS THE POINT. A base managed SFT does not accept fails at submission - after the
# dataset is written, uploaded and paid for. The kit's tune.py refuses before any of that, and its list is
# DATED: gemini-3.5-flash and gemini-3.1-flash-lite as of 4 September 2026; gemini-3.6-flash, the course
# default for inference, is not tunable. Re-read the tuning page before the room and update TUNABLE.
print("tunable as of", kit_tune.TUNABLE_DATE, ":", sorted(kit_tune.TUNABLE))
try:
    kit_tune.launch(PROJECT_ID, f"gs://{DATASETS}/sft/documind_sft_v1.vertex.jsonl", "gemini-3.6-flash", 3, 4, "x")
    raise AssertionError("an untunable base was accepted")
except SystemExit as e:
    print("refused before submission:", str(e)[:120], "...")

BASE = "gemini-3.1-flash-lite"    # D1: the cheapest tunable base, and the tier the router already sends simple questions to
if LAUNCH:
    dataset_uri = f"gs://{DATASETS}/sft/documind_sft_{'v1' if SOURCE == 'frozen' else 'colab'}.vertex.jsonl"
    if SOURCE == "colab":
        gcs.bucket(DATASETS).blob("sft/documind_sft_colab.vertex.jsonl").upload_from_filename("documind_sft.vertex.jsonl")
    job = kit_tune.launch(PROJECT_ID, dataset_uri, BASE, EPOCHS, ADAPTER, f"documind-sft-{SOURCE}")
    print("submitted:", job.name, "- poll it with: python evals/tune.py --project", PROJECT_ID, "--poll", job.name)
else:
    print(f"\nLAUNCH=False. From deploy/: make tune PROJECT={PROJECT_ID} TUNE_BASE={BASE}   (it waits, then prints the endpoint)")


## Cell 6: The model is a setting


In [ ]:
import inspect
import generator as api_generator          # the deployed API's own module, from the clone

# THE MODEL IS A SETTING. GENERATOR_MODEL names what answers: a model name is served on the global endpoint,
# a tuned model is an ENDPOINT PATH in the region it was tuned in, and the generator picks the client by
# the value. Every surface above the API - the MCP server, the agents, the UI - changes nothing when the
# value changes. That is what makes a tuned model an A/B and not a migration.
print(inspect.getsource(api_generator._client_for))
print("the usage row names the model that answered:", usage_rows(minutes=60, limit=1)[0].get("model") if usage_rows(minutes=60, limit=1) else "(no rows in the last hour - ask a question first)")


## Cell 7: The A/B that counts - the gate, twice
The nine thresholds on the live revision, then on the candidate revision with the tuned endpoint behind it. One dataset, one number per threshold.


In [ ]:
from run_eval import live

# THE A/B THAT COUNTS: THE GATE, TWICE. run_eval.live() is the nine thresholds - answerable, cited, the figure
# present, refusals, isolation - over 65 rows, and it reads its tokens from the environment the way
# `make eval-live` sets them. First the live revision; then, when TUNED_ENDPOINT is set, the candidate
# revision that `make candidate` tagged with no traffic and the tuned endpoint behind it. Two eval lines,
# one dataset, one number per threshold: that is a tuning decision with evidence.
def eval_line(api_url: str) -> int:
    # The token's audience is the SERVICE's canonical URL, whichever of its URLs is called: the API verifies
    # bearer tokens against SELF_URL, and a token minted for the candidate's tag URL is refused with 401 on
    # every row (F42, the candidate's first eval). One audience, two URLs.
    os.environ["DOCUMIND_ID_TOKEN"] = id_token_as(MEMBER_SA, API_URL)
    os.environ["DOCUMIND_OUTSIDER_TOKEN"] = id_token_as(OUTSIDER_SA, API_URL)
    return live(api_url)

print("== live revision ==")
rc = eval_line(API_URL)
assert rc == 0, "the live revision does not clear the gate - fix that before comparing anything to it"

if TUNED_ENDPOINT:
    CANDIDATE_URL = f"https://candidate---documind-api-{NUMBER}.{REGION}.run.app"
    print(f"\n== candidate revision ({TUNED_ENDPOINT.rsplit('/', 1)[-1]}) ==")
    print(f"(make candidate PROJECT={PROJECT_ID} GENERATOR_MODEL={TUNED_ENDPOINT} RAG_MODEL_BASE={BASE} must have run)")
    rc_b = eval_line(CANDIDATE_URL)
    print("\nthe tuned model", "clears the gate" if rc_b == 0 else "does NOT clear the gate", "- and 10.4's judge says by how much")
else:
    print("\nTUNED_ENDPOINT is empty: run make tune, then make candidate with the endpoint it prints, paste it above, re-run")


## Cell 8: Fine-tune or prompt, with the lane's numbers


In [ ]:
# FINE-TUNE OR PROMPT? The framework, with the lane's own numbers in it. Every input here is something the
# lane measured today: the rows in the file, the format the contract already fixes, the monthly volume
# from the usage rows, and whether the reason is one of the four 10.5 accepts (format, latency, cost,
# residency) - facts and freshness are retrieval's job.
def should_finetune(num_examples, monthly_calls, format_consistency_needed, domain_specific_terms, requirements_stable, reason):
    score, reasons = 0, []
    if num_examples >= 100: score += 2; reasons.append(f"+ {num_examples} examples (>= 100)")
    else: reasons.append(f"- only {num_examples} examples (< 100): few-shot instead")
    if monthly_calls > 50_000: score += 2; reasons.append(f"+ {monthly_calls:,} calls a month: few-shot tokens add up")
    if format_consistency_needed: score += 2; reasons.append("+ the answer must always be ModelDraft's JSON")
    if domain_specific_terms: score += 1; reasons.append("+ Indian statute vocabulary")
    if not requirements_stable: score -= 3; reasons.append("- the contract still moves: prompts are faster to change")
    if reason not in ("format", "latency", "cost", "residency"): score -= 4; reasons.append(f"- '{reason}' is retrieval's job, not tuning's")
    return ("FINE-TUNE" if score >= 3 else "PROMPT"), score, reasons

rows_last_month = len(usage_rows(minutes=60 * 24 * 30, limit=1000))
rec, score, why = should_finetune(len(examples), max(rows_last_month, 1) * 30, True, True, True, "format")
print(rec, f"(score {score})"); [print("  ", w) for w in why]


## Where this goes
- **10.4** is the second judge on the same answers: groundedness and fulfilment by Gemini, pairwise base against tuned, an Experiments run per commit.
- **10.5** tunes a model you keep, on the SAME file in its chat format, on a free T4.

## ✅ Lesson 10.1 complete
- ✅ A dataset from the corpus, in the lane's grammar, the golden set excluded, DLP clean, a manifest with a sha
- ✅ The two rules re-run on the file; the price from measured tokens
- ✅ A base the service refuses, refused before submission; the launch behind a switch
- ✅ The model as a setting: one variable, a client per value, nothing above the API changes
- ✅ The gate twice: the live revision and the candidate
